In [13]:
import os
import numpy as np
import importlib
import pandas as pd
import obj_2_pcd
importlib.reload(obj_2_pcd)

tiff_dir_root = '/data/jhahn/data/brain_lightsheet/slices'
obj_dir_root = '/data/jhahn/data/shape_dataset/data/brain_lightsheet_3'

dataset_annotation_file_name = obj_dir_root+"/data.csv"

_df  = obj_2_pcd.create_dataset(dataset_annotation_file_name, tiff_dir_root, obj_dir_root)
_df.head(n=10)
#data_loader = obj_2_pcd.create_data_loader(dataset_annotation_file_name)
#print(f"✅ DataLoader 생성 완료. 총 배치 개수: {len(data_loader)}")
#_df.head()

,obj_dir_root,image_filename_list_for_one_sub_brain,slice_angle,tickness,from_index,to_index,num_of_missing_slices_dist,num_of_missing_slices_list,is_curvature,num_of_slices
0,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,66,Merged,"8,4,4,4,4,4,4,4,4,4",False,10
1,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,66,Merged,"8,4,4,4,4,4,4,4,4,4",True,10
2,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,60,Merged,"3,3,3,6,12,3,3,3",False,8
3,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,60,Merged,"3,3,3,6,12,3,3,3",True,8
4,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,65,Merged,"3,3,6,6,12,6,3",False,7
5,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,65,Merged,"3,3,6,6,12,6,3",True,7
6,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,56,Merged,"2,4,4,2,6,6,2,2",False,8
7,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,56,Merged,"2,4,4,2,6,6,2,2",True,8
8,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,60,Merged,"2,4,4,6,2,2,2,4,4",False,9
9,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,60,Merged,"2,4,4,6,2,2,2,4,4",True,9


,obj_dir_root,image_filename_list_for_one_sub_brain,slice_angle,tickness,from_index,to_index,num_of_missing_slices_dist,num_of_missing_slices_list,is_curvature,num_of_slices
0,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,39,Merged,"19,16",False,2
1,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,39,Merged,"19,16",True,2
2,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,64,Merged,"14,20,24",False,3
3,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,64,Merged,"14,20,24",True,3
4,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.003,0,67,Merged,"14,8,16,21",False,4


In [3]:
import torch
import pandas as pd
import numpy as np
import torch.multiprocessing as mp
from tqdm import tqdm
import importlib
import SlicedVolumeDataset
importlib.reload(SlicedVolumeDataset)

import obj_2_pcd
importlib.reload(obj_2_pcd)
# 데이터 변환 (예시)


if torch.cuda.is_available():
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")


### 2. DataLoader 순회 (Iteration)
tasks_to_run = []
# 일반적으로 훈련 루프(Training Loop)에서 사용됩니다.


# DataLoader를 순회하며 배치 단위로 데이터(이미지)와 레이블을 가져옵니다.
for batch_idx, (tiff_images, labels, output_dir) in enumerate(data_loader):
    
    missing_slices_list = [t.item() for t in labels['missing_slices_list']]
    _tiff_images = [t[0] for t in tiff_images]
    _output_dir = output_dir[0]


    #print((_tiff_images, labels['tickness']
    #        ,missing_slices_list, _output_dir, labels['num_of_slices'].item(), labels['is_curvature'].item(),'glb', device))
    tasks_to_run.append((_tiff_images, labels['tickness'].item()
            ,missing_slices_list, _output_dir, labels['num_of_slices'].item(), labels['is_curvature'].item(),'glb' ))
    
    if len(tasks_to_run) > 10:
        break
mp.set_start_method('spawn', force=True)
with mp.Pool( ) as pool: # Use a pool of 4 processes
    pool.starmap(obj_2_pcd.tiff_lilst_2_brain_obj, tqdm(tasks_to_run, total=len(tasks_to_run), desc="_tiff_2_pcd_func"))

print("DONE!")

_tiff_2_pcd_func: 100%|██████████| 11/11 [00:00<00:00, 1546.00it/s]


DONE!
